# API Tools

For this section we'll just get a default ReAct agent to run our tools; this will let us focus our debugging in the tools themselves rather than the agent.

In [11]:
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [1]:
!uv pip install python-dotenv requests

from dotenv import load_dotenv, find_dotenv
import requests, os
from urllib.parse import urljoin
from typing import Literal

# Add pydantic for response validation
!uv pip install pydantic
from pydantic import BaseModel
from typing import Any, Optional

class Response(BaseModel):
    """Base response class with validation."""
    pass

class ValidResponse(Response):
    """Valid response with data."""
    data: Any

class ErrorResponse(Response):
    """Error response with error message."""
    error: str
    status_code: Optional[int] = None

# Load environment variables from .env file
load_dotenv(find_dotenv())
FOOTBALL_API_KEY = os.getenv('FOOTBALL_API_KEY')
FOOTBALL_API_BASE_URL = "https://v3.football.api-sports.io/"
FOOTBALL_API_HOST = "v3.football.api-sports.io"

def check_api_response_status(response: requests.Response) -> dict:
    """
    Generic status check for API responses.
    
    Args:
        response: The requests.Response object from an API call
        
    Returns:
        Dictionary with success status and error information if applicable
    """
    if response.status_code == 200:
        try:
            data = response.json()
            return {"success": True, "data": data}
        except requests.exceptions.JSONDecodeError:
            return {
                "success": False, 
                "error": "Invalid JSON response",
                "status_code": response.status_code
            }
    elif response.status_code == 204:
        return {
            "success": False,
            "error": "No content available",
            "status_code": response.status_code
        }
    elif response.status_code == 499:
        return {
            "success": False,
            "error": "Request timeout",
            "status_code": response.status_code
        }
    elif response.status_code == 500:
        return {
            "success": False,
            "error": "Internal server error",
            "status_code": response.status_code
        }
    else:
        return {
            "success": False,
            "error": f"HTTP {response.status_code}: {response.reason}",
            "status_code": response.status_code
        }

def call_football_api(method: Literal["GET", "OPTIONS", "HEAD", "POST", "PUT", "PATCH", "DELETE"], endpoint: str, data={}, params=None) -> Response:
    """
    Call the Football API with the given endpoint and parameters.
    Now includes proper status code checking and returns Response objects.
    
    Args:
        method: HTTP method to use
        endpoint: API endpoint to call
        data: Data to send in the request body
        params: Query parameters
        
    Returns:
        ValidResponse with data or ErrorResponse with error information
    """
    url = urljoin(FOOTBALL_API_BASE_URL, endpoint)
    headers = {
        'x-rapidapi-key': FOOTBALL_API_KEY,
        'x-rapidapi-host': FOOTBALL_API_HOST
    }
    
    try:
        response = requests.request(method, url, headers=headers, data=data, params=params)
        result = check_api_response_status(response)
        
        if result["success"]:
            return ValidResponse(data=result["data"])
        else:
            logger.error(f"API call failed: {result['error']}")
            return ErrorResponse(
                error=result["error"], 
                status_code=result.get("status_code")
            )
            
    except requests.exceptions.RequestException as e:
        logger.error(f"Request failed: {str(e)}")
        return ErrorResponse(error=f"Request failed: {str(e)}")

Audited 2 packages in 3ms
Audited 1 package in 3ms
Audited 1 package in 3ms


In [9]:
# Test the API functions
print("Testing API status...")
status_response = call_football_api("GET", "status")
print(f"Status response: {status_response}")

print("\nTesting error handling with invalid endpoint...")
error_response = call_football_api("GET", "invalid_endpoint")
print(f"Error response: {error_response}")

Testing API status...
Status response: data={'get': 'status', 'parameters': [], 'errors': [], 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': {'account': {'firstname': 'Vasco', 'lastname': 'Peleteiro', 'email': 'vasco@augustalabs.ai'}, 'subscription': {'plan': 'Pro', 'end': '2025-07-05T20:58:08+00:00', 'active': True}, 'requests': {'current': 49, 'limit_day': 7500}}}

Testing error handling with invalid endpoint...
Error response: data={'get': 'invalid_endpoint', 'parameters': [], 'errors': {'endpoint': 'The Invalid_endpoint endpoint does not exist.'}, 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': []}
Status response: data={'get': 'status', 'parameters': [], 'errors': [], 'results': 0, 'paging': {'current': 1, 'total': 1}, 'response': {'account': {'firstname': 'Vasco', 'lastname': 'Peleteiro', 'email': 'vasco@augustalabs.ai'}, 'subscription': {'plan': 'Pro', 'end': '2025-07-05T20:58:08+00:00', 'active': True}, 'requests': {'current': 49, 'limit_day': 7

We need to implement a tool that can query the [Football API](https://www.api-football.com/documentation-v3) to get information about football leagues, teams, players, and matches.

1. Classificações de equipas — <https://www.api-football.com/documentation-v3#tag/Standings/operation/get-standings>
2. Próximos jogos — <https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `next` parameter
3. Últimos jogos — <https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `last` parameter
4. Jogos específicos (ex: SLB vs SCP para a liga em 2012/13) — we need to get the fixture ID first, then use it to get the match details.
5. Resultados de jogos específicos
6. Eventos de jogos específicos (ex.: golos, cartões, substituições).
7. Estatísticas de jogadores (ex: número de golos do jogador X na época Y)

Qualquer equipa, jogo ou jogador das top 7 ligas europeias + das 3 competições europeias.

Extra:

8. Odds
9. H2H
10. ...


## API Football

There doesn't seem to be a OpenAPI spec for this API, but we'll use the documentation to implement the tools we need and see how it goes.

### 0. Setup

To start we'll need to get the IDs for all leagues and teams for the top 7 leagues in Europe, as well as the European competitions. They are:
- Premier League (England)
- La Liga (Spain)
- Serie A (Italy)
- Bundesliga (Germany)
- Ligue 1 (France)
- Primeira Liga (Portugal)
- Eredivisie (Netherlands)
- UEFA Champions League
- UEFA Europa League
- UEFA Conference League

We should get the cups as well:
- FA Cup (England)
- Copa del Rey (Spain)
- Coppa Italia (Italy)
- DFB Pokal (Germany)
- Coupe de France (France)
- Taça de Portugal (Portugal)
- KNVB Beker (Netherlands)
- UEFA Super Cup

We'll save these in a JSON dictionary to save on API calls later and define local functions to retrieve them.

In [10]:
leagues_en = call_football_api("GET", "leagues", params={"search": "England"})

In [11]:
leagues_en

ValidResponse(data={'get': 'leagues', 'parameters': {'search': 'England'}, 'errors': [], 'results': 46, 'paging': {'current': 1, 'total': 1}, 'response': [{'league': {'id': 39, 'name': 'Premier League', 'type': 'League', 'logo': 'https://media.api-sports.io/football/leagues/39.png'}, 'country': {'name': 'England', 'code': 'GB-ENG', 'flag': 'https://media.api-sports.io/flags/gb-eng.svg'}, 'seasons': [{'year': 2010, 'start': '2010-08-14', 'end': '2011-05-17', 'current': False, 'coverage': {'fixtures': {'events': True, 'lineups': True, 'statistics_fixtures': False, 'statistics_players': False}, 'standings': True, 'players': True, 'top_scorers': True, 'top_assists': True, 'top_cards': True, 'injuries': False, 'predictions': True, 'odds': False}}, {'year': 2011, 'start': '2011-08-13', 'end': '2012-05-13', 'current': False, 'coverage': {'fixtures': {'events': True, 'lineups': True, 'statistics_fixtures': False, 'statistics_players': False}, 'standings': True, 'players': True, 'top_scorers': 

In [3]:
leagues = call_football_api("GET", "leagues", params={})
leagues

ValidResponse(data={'get': 'leagues', 'parameters': [], 'errors': [], 'results': 1186, 'paging': {'current': 1, 'total': 1}, 'response': [{'league': {'id': 4, 'name': 'Euro Championship', 'type': 'Cup', 'logo': 'https://media.api-sports.io/football/leagues/4.png'}, 'country': {'name': 'World', 'code': None, 'flag': None}, 'seasons': [{'year': 2008, 'start': '2008-06-07', 'end': '2008-06-29', 'current': False, 'coverage': {'fixtures': {'events': True, 'lineups': True, 'statistics_fixtures': False, 'statistics_players': False}, 'standings': False, 'players': True, 'top_scorers': True, 'top_assists': True, 'top_cards': True, 'injuries': False, 'predictions': True, 'odds': False}}, {'year': 2012, 'start': '2012-06-08', 'end': '2012-07-01', 'current': False, 'coverage': {'fixtures': {'events': True, 'lineups': True, 'statistics_fixtures': False, 'statistics_players': False}, 'standings': False, 'players': True, 'top_scorers': True, 'top_assists': True, 'top_cards': True, 'injuries': False, 

In [5]:
def extract_league_info(leagues_response):
    """
    Extract league name, id, and country from the Football API leagues response.
    
    Args:
        leagues_response: ValidResponse object or dict from the Football API leagues endpoint
        
    Returns:
        List of dictionaries containing league info
    """
    league_info = []
    
    # Handle both ValidResponse objects and plain dictionaries
    if isinstance(leagues_response, ValidResponse):
        data = leagues_response.data
    else:
        data = leagues_response
    
    if 'response' in data:
        for item in data['response']:
            league = item.get('league', {})
            country = item.get('country', {})
            
            league_data = {
                'id': league.get('id'),
                'name': league.get('name'),
                'country': country.get('name')
            }
            league_info.append(league_data)
    
    return league_info

# Test
leagues_parsed = extract_league_info(leagues)
for league in leagues_parsed:
    print(f"ID: {league['id']}, Name: {league['name']}, Country: {league['country']}")

ID: 4, Name: Euro Championship, Country: World
ID: 21, Name: Confederations Cup, Country: World
ID: 61, Name: Ligue 1, Country: France
ID: 144, Name: Jupiler Pro League, Country: Belgium
ID: 71, Name: Serie A, Country: Brazil
ID: 39, Name: Premier League, Country: England
ID: 78, Name: Bundesliga, Country: Germany
ID: 135, Name: Serie A, Country: Italy
ID: 88, Name: Eredivisie, Country: Netherlands
ID: 94, Name: Primeira Liga, Country: Portugal
ID: 140, Name: La Liga, Country: Spain
ID: 179, Name: Premiership, Country: Scotland
ID: 180, Name: Championship, Country: Scotland
ID: 1, Name: World Cup, Country: World
ID: 803, Name: Asian Games, Country: World
ID: 804, Name: Caribbean Cup, Country: World
ID: 62, Name: Ligue 2, Country: France
ID: 2, Name: UEFA Champions League, Country: World
ID: 311, Name: 1st Division, Country: Albania
ID: 310, Name: Superliga, Country: Albania
ID: 186, Name: Ligue 1, Country: Algeria
ID: 187, Name: Ligue 2, Country: Algeria
ID: 42, Name: League Two, Count

Let's extract the leagues we want:

In [6]:
# Define the leagues we want with their countries
target_leagues = {
  "Premier League": "England",
  "La Liga": "Spain", 
  "Serie A": "Italy",
  "Bundesliga": "Germany",
  "Ligue 1": "France",
  "Primeira Liga": "Portugal",
  "Eredivisie": "Netherlands",
  "UEFA Champions League": "World",
  "UEFA Europa League": "World",
  "UEFA Europa Conference League": "World",
  "FA Cup": "England",
  "Copa del Rey": "Spain",
  "Coppa Italia": "Italy",
  "DFB Pokal": "Germany",
  "Coupe de France": "France",
  "Taça de Portugal": "Portugal",
  "KNVB Beker": "Netherlands",
  "UEFA Super Cup": "World"
}

selected_leagues_filtered = []

for league in leagues_parsed:
  league_name = league['name']
  league_country = league['country']
  
  if league_name in target_leagues and target_leagues[league_name] == league_country:
    selected_leagues_filtered.append(league)
    print(f"Selected League: {league_name} (ID: {league['id']}, Country: {league_country})")

print(f"\nTotal selected leagues: {len(selected_leagues_filtered)}")

Selected League: Ligue 1 (ID: 61, Country: France)
Selected League: Premier League (ID: 39, Country: England)
Selected League: Bundesliga (ID: 78, Country: Germany)
Selected League: Serie A (ID: 135, Country: Italy)
Selected League: Eredivisie (ID: 88, Country: Netherlands)
Selected League: Primeira Liga (ID: 94, Country: Portugal)
Selected League: La Liga (ID: 140, Country: Spain)
Selected League: UEFA Champions League (ID: 2, Country: World)
Selected League: Coupe de France (ID: 66, Country: France)
Selected League: FA Cup (ID: 45, Country: England)
Selected League: DFB Pokal (ID: 81, Country: Germany)
Selected League: UEFA Europa League (ID: 3, Country: World)
Selected League: UEFA Super Cup (ID: 531, Country: World)
Selected League: Taça de Portugal (ID: 96, Country: Portugal)
Selected League: Coppa Italia (ID: 137, Country: Italy)
Selected League: KNVB Beker (ID: 90, Country: Netherlands)
Selected League: Copa del Rey (ID: 143, Country: Spain)
Selected League: UEFA Europa Conferen

In [7]:
# Save the selected leagues to a dictionary mapping the name to the ID
selected_leagues_dict = {league['name']: league['id'] for league in selected_leagues_filtered}
print(selected_leagues_dict)
# Save the selected leagues to a JSON file
import json
with open('top_leagues.json', 'w') as f:
    json.dump(selected_leagues_dict, f, indent=4)
print("Selected leagues saved to 'top_leagues.json'")

{'Ligue 1': 61, 'Premier League': 39, 'Bundesliga': 78, 'Serie A': 135, 'Eredivisie': 88, 'Primeira Liga': 94, 'La Liga': 140, 'UEFA Champions League': 2, 'Coupe de France': 66, 'FA Cup': 45, 'DFB Pokal': 81, 'UEFA Europa League': 3, 'UEFA Super Cup': 531, 'Taça de Portugal': 96, 'Coppa Italia': 137, 'KNVB Beker': 90, 'Copa del Rey': 143, 'UEFA Europa Conference League': 848}
Selected leagues saved to 'top_leagues.json'


In [8]:
# Define function to get league ID by name
def get_league_id_by_name(league_name: str) -> int | None:
    """
    Get the league ID by its name.

    Args:
        league_name: Name of the league

    Returns:
        League ID if found, otherwise None
    """
    try:
        leagues = json.load(open("top_leagues.json"))
    except FileNotFoundError:
        logger.warning("top_leagues.json not found. Will fetch from API...")
        leagues = {}

    # Exact match first
    if league_name in leagues:
        return leagues[league_name]

    # If the league is not found, partial match the name
    #for league_key in leagues:
    #    if league_name.lower() in league_key.lower() or league_key.lower() in league_name.lower():
    #        return leagues[league_key]

    # If no match is found, fetch it from the API:
    logger.warning(
        f"League '{league_name}' not found in selected leagues. Fetching from API..."
    )
    response = call_football_api("GET", "leagues", params={"search": league_name})
    if isinstance(response, ValidResponse):
        leagues_parsed = extract_league_info(response.data)
        for league in leagues_parsed:
            if league["name"].lower() in league_name.lower() or league_name.lower() in league["name"].lower():
                return league["id"]
        logger.error(f"League '{league_name}' not found in API response.")

    return None


def list_leagues() -> list[str]:
    """
    List all available leagues.

    Returns:
        List of league names
    """
    try:
        leagues = json.load(open("top_leagues.json"))
        return list(leagues.keys())
    except FileNotFoundError:
        logger.warning("top_leagues.json not found.")
        return []

In [9]:
# Test exact league match
league_name = "Premier League"
league_id = get_league_id_by_name(league_name)
print(f"League: {league_name}, ID: {league_id}")

League: Premier League, ID: 39


In [12]:
# Test partial match
league_name_partial = "Premier"
league_id_partial = get_league_id_by_name(league_name_partial)
print(f"League (partial): {league_name_partial}, ID: {league_id_partial}")

League (partial): Premier, ID: 39


In [13]:
# Test list leagues function
available_leagues = list_leagues()
print(f"Available leagues: {available_leagues[:5]}...")  # Show first 5

Available leagues: ['Ligue 1', 'Premier League', 'Bundesliga', 'Serie A', 'Eredivisie']...


In [14]:
# Test with a league not in the list
league_name_not_found = "Nonexistent League"
league_id_not_found = get_league_id_by_name(league_name_not_found)
print(f"League (not found): {league_name_not_found}, ID: {league_id_not_found}")

League (not found): Nonexistent League, ID: None


In [15]:
# Test with a league not in the list that exists in the API
league_name_exists = "Jupiler Pro League"
league_id_exists = get_league_id_by_name(league_name_exists)
print(f"League (exists in API): {league_name_exists}, ID: {league_id_exists}")

League (exists in API): Jupiler Pro League, ID: 144


Now we'll do the same for the teams. We'll cache the teams in the latest season of each league and keep a fallback method to get the teams for a specific season if needed.

In [ ]:
# Get all teams from one {league} & {season}
# get("https://v3.football.api-sports.io/teams?league=39&season=2019");
# Allows you to search for a team in relation to a team {name} or {country}
# get("https://v3.football.api-sports.io/teams?search=manches");
# get("https://v3.football.api-sports.io/teams?search=England");

# Get all teams from the top leagues and save to top_teams.json
import json
import time

league_ids = json.load(open('top_leagues.json'))
all_teams = {}

for league_name, league_id in league_ids.items():
    season = 2024
    print(f"Fetching teams for {league_name} (ID: {league_id})...")
    
    teams_response = call_football_api("GET", "teams", params={"league": league_id, "season": season})
    
    # Handle ValidResponse object properly
    if isinstance(teams_response, ValidResponse):
        print(f"Found {len(teams_response.data['response'])} teams in {league_name}")
        for team_data in teams_response.data['response']:
            team = team_data['team']
            team_name = team['name']
            team_id = team['id']
            
            # Add team to dictionary (team name as key, ID as value)
            all_teams[team_name] = team_id
            print(f"  - {team_name} (ID: {team_id})")
    else:
        print(f"No teams found for {league_name} (ID: {league_id}) in season {season}")
        if isinstance(teams_response, ErrorResponse):
            print(f"Error: {teams_response.error}")
    
    # Small delay to not hammer the API
    time.sleep(0.1)

# Save all teams to JSON file
with open('top_teams.json', 'w') as f:
    json.dump(all_teams, f, indent=4, ensure_ascii=False)

print(f"\nTotal teams saved: {len(all_teams)}")
print("Teams saved to 'top_teams.json'")

Fetching teams for Ligue 1 (ID: 61)...
Found 19 teams in Ligue 1
  - Angers (ID: 77)
  - Lille (ID: 79)
  - Lyon (ID: 80)
  - Marseille (ID: 81)
  - Montpellier (ID: 82)
  - Nantes (ID: 83)
  - Nice (ID: 84)
  - Paris Saint Germain (ID: 85)
  - Monaco (ID: 91)
  - Reims (ID: 93)
  - Rennes (ID: 94)
  - Strasbourg (ID: 95)
  - Toulouse (ID: 96)
  - Stade Brestois 29 (ID: 106)
  - Auxerre (ID: 108)
  - Le Havre (ID: 111)
  - Metz (ID: 112)
  - Lens (ID: 116)
  - Saint Etienne (ID: 1063)
Fetching teams for Premier League (ID: 39)...
Found 20 teams in Premier League
  - Manchester United (ID: 33)
  - Newcastle (ID: 34)
  - Bournemouth (ID: 35)
  - Fulham (ID: 36)
  - Wolves (ID: 39)
  - Liverpool (ID: 40)
  - Southampton (ID: 41)
  - Arsenal (ID: 42)
  - Everton (ID: 45)
  - Leicester (ID: 46)
  - Tottenham (ID: 47)
  - West Ham (ID: 48)
  - Chelsea (ID: 49)
  - Manchester City (ID: 50)
  - Brighton (ID: 51)
  - Crystal Palace (ID: 52)
  - Brentford (ID: 55)
  - Ipswich (ID: 57)
  - Nottin

In [16]:
def get_team_id_by_name(team_name: str) -> int | None:
    """
    Get the team ID by its name.

    Args:
        team_name: Name of the team

    Returns:
        Team ID if found, otherwise None
    """
    try:
        teams = json.load(open("top_teams.json"))
    except FileNotFoundError:
        logger.warning("top_teams.json not found. Will fetch from API...")
        teams = {}

    # Exact match first
    if team_name in teams:
        return teams[team_name]

    # If the team is not found, try partial match
    for team_key, team_id in teams.items():
        if team_name.lower() in team_key.lower() or team_key.lower() in team_name.lower():
            return team_id

    # If no match is found, fetch it from the API:
    logger.warning(
        f"Team '{team_name}' not found in cached teams. Fetching from API..."
    )
    response = call_football_api("GET", "teams", params={"search": team_name})

    if isinstance(response, ValidResponse):
        for team_data in response.data["response"]:
            team = team_data["team"]
            if team["name"].lower() == team_name.lower():
                return team["id"]

    return None


def list_teams() -> list[str]:
    """
    List all available teams from cached data.

    Returns:
        List of team names
    """
    try:
        teams = json.load(open("top_teams.json"))
        return list(teams.keys())
    except FileNotFoundError:
        logger.warning("top_teams.json not found.")
        return []



In [20]:
# Test exact team name match
team_name = "Manchester United"
team_id = get_team_id_by_name(team_name)
print(f"Team: {team_name}, ID: {team_id}")

Team: Manchester United, ID: 33


In [21]:
# Test partial team name match
team_name_partial = "Arsenal"
team_id_partial = get_team_id_by_name(team_name_partial)
print(f"Team (partial): {team_name_partial}, ID: {team_id_partial}")

Team (partial): Arsenal, ID: 42


In [22]:
# Test with a team not in the list
team_name_not_found = "Nonexistent Team"
team_id_not_found = get_team_id_by_name(team_name_not_found)
print(f"Team (not found): {team_name_not_found}, ID: {team_id_not_found}")

Team (not found): Nonexistent Team, ID: None


In [23]:
# Test with a team that exists in the API but not in the cached data
team_name_exists = "Fluminense"
team_id_exists = get_team_id_by_name(team_name_exists)
print(f"Team (exists in API): {team_name_exists}, ID: {team_id_exists}")

Team (exists in API): Fluminense, ID: 124


In [24]:
# Test list teams function
available_teams = list_teams()
print(f"Total teams available: {len(available_teams)}")
print(f"First 5 teams: {available_teams[:5]}")  # Show first 5

Total teams available: 1622
First 5 teams: ['Angers', 'Lille', 'Lyon', 'Marseille', 'Montpellier']


## 1. Team standings

<https://www.api-football.com/documentation-v3#tag/Standings/operation/get-standings>

In [35]:
def get_standings(league_name: str, season: int, team_name: str | None = None) -> Response:
    """
    Get the standings for a league or specific team in a league.
    
    Args:
        league_name: Name of the league (e.g., "Premier League", "La Liga")
        season: Season year (4 digits, e.g., 2024)
        team_name: Optional team name to filter standings for specific team
        
    Returns:
        ValidResponse with standings data or ErrorResponse with error details
    """
    # Get league ID from the league name
    league_id = get_league_id_by_name(league_name)
    if league_id is None:
        logger.error(f"League '{league_name}' not found")
        return ErrorResponse(error=f"League '{league_name}' not found")
    
    # Prepare parameters for the API call
    params = {
        "league": league_id,
        "season": season
    }
    
    # If team name is provided, get team ID and add to params
    if team_name:
        team_id = get_team_id_by_name(team_name)
        if team_id is None:
            logger.error(f"Team '{team_name}' not found")
            return ErrorResponse(error=f"Team '{team_name}' not found")
        params["team"] = team_id
    
    logger.info(f"Fetching standings for {league_name} (ID: {league_id}) season {season}")
    if team_name:
        logger.info(f"Filtering for team: {team_name}")
    
    # Make the API call
    try:
        response = call_football_api("GET", "standings", params=params)
        
        if isinstance(response, ValidResponse) and "response" in response.data and response.data["response"]:
            logger.info(f"Successfully retrieved standings data")
            return response
        elif isinstance(response, ValidResponse):
            logger.warning(f"No standings data found for {league_name} season {season}")
            return ErrorResponse(error=f"No standings data found for {league_name} season {season}")
        else:
            return response  # Already an ErrorResponse
            
    except Exception as e:
        logger.error(f"Error fetching standings: {str(e)}")
        return ErrorResponse(error=f"Error fetching standings: {str(e)}")


def format_standings_table(standings_response: Response) -> str:
    """
    Format the standings response into a readable table string.
    
    Args:
        standings_response: Response from get_standings function
        
    Returns:
        Formatted string table of the standings
    """
    if not isinstance(standings_response, ValidResponse):
        return f"Error: {standings_response.error}" # type: ignore
    
    data = standings_response.data
    if "response" not in data or not data["response"]:
        return "No standings data available"
    
    # Extract standings data
    standings_data = data["response"][0]["league"]["standings"][0]
    
    # Create table header
    table = f"{'Pos':<4} {'Team':<25} {'GP':<3} {'W':<3} {'D':<3} {'L':<3} {'GF':<3} {'GA':<3} {'GD':<4} {'Pts':<4}\n"
    table += "-" * 75 + "\n"
    
    # Add each team's data
    for team_data in standings_data:
        rank = team_data["rank"]
        team_name = team_data["team"]["name"]
        all_stats = team_data["all"]
        
        played = all_stats["played"]
        wins = all_stats["win"]
        draws = all_stats["draw"]
        losses = all_stats["lose"]
        goals_for = all_stats["goals"]["for"]
        goals_against = all_stats["goals"]["against"]
        goal_diff = goals_for - goals_against
        points = team_data["points"]
        
        # Truncate team name if too long
        display_name = team_name[:24] if len(team_name) > 24 else team_name
        
        table += f"{rank:<4} {display_name:<25} {played:<3} {wins:<3} {draws:<3} {losses:<3} {goals_for:<3} {goals_against:<3} {goal_diff:<4} {points:<4}\n"
    
    return table


First we test the function:

In [ ]:
test_standings = get_standings("Premier League", 2024)
print("Raw API Response:")
print(test_standings)
print("\n" + "="*50 + "\n")
print("Formatted Table:")
print(format_standings_table(test_standings))

INFO:__main__:Fetching standings for Premier League (ID: 39) season 2024
INFO:__main__:Successfully retrieved standings data


Raw API Response:
data={'get': 'standings', 'parameters': {'league': '39', 'season': '2024'}, 'errors': [], 'results': 1, 'paging': {'current': 1, 'total': 1}, 'response': [{'league': {'id': 39, 'name': 'Premier League', 'country': 'England', 'logo': 'https://media.api-sports.io/football/leagues/39.png', 'flag': 'https://media.api-sports.io/flags/gb-eng.svg', 'season': 2024, 'standings': [[{'rank': 1, 'team': {'id': 40, 'name': 'Liverpool', 'logo': 'https://media.api-sports.io/football/teams/40.png'}, 'points': 84, 'goalsDiff': 45, 'group': 'Premier League', 'form': 'DLDLW', 'status': 'same', 'description': 'Champions League', 'all': {'played': 38, 'win': 25, 'draw': 9, 'lose': 4, 'goals': {'for': 86, 'against': 41}}, 'home': {'played': 19, 'win': 14, 'draw': 4, 'lose': 1, 'goals': {'for': 42, 'against': 16}}, 'away': {'played': 19, 'win': 11, 'draw': 5, 'lose': 3, 'goals': {'for': 44, 'against': 25}}, 'update': '2025-05-26T00:00:00+00:00'}, {'rank': 2, 'team': {'id': 42, 'name': 'Arse

Then we convert it into a tool and create a ReAct agent to use it.

In [47]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class GetStandingsInput(BaseModel):
    league_name: str = Field(..., description="Name of the league (e.g., 'Premier League', 'La Liga')")
    season: int = Field(..., description="Season year (4 digits, e.g., 2024)")
    team_name: Optional[str] = Field(None, description="Optional team name to filter standings for specific team")

# create a tool from the get_standings function
get_standings_tool = StructuredTool.from_function(
    get_standings,
    name="get_standings",
    description="Get the standings for a league or specific team in a league.",
    args_schema=GetStandingsInput,
    return_direct=False
)

print(get_standings_tool.name)
print(get_standings_tool.description)
print(get_standings_tool.args)

get_standings
Get the standings for a league or specific team in a league.
{'league_name': {'description': "Name of the league (e.g., 'Premier League', 'La Liga')", 'title': 'League Name', 'type': 'string'}, 'season': {'description': 'Season year (4 digits, e.g., 2024)', 'title': 'Season', 'type': 'integer'}, 'team_name': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional team name to filter standings for specific team', 'title': 'Team Name'}}


In [38]:
!uv pip install -qU "langchain[google-genai]"

import getpass
import os
from langchain_google_genai import ChatGoogleGenerativeAI

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

model = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [48]:
# Create a ReAct agent to use the tool
from langgraph.prebuilt import create_react_agent

react_agent = create_react_agent(
    model=model,
    tools=[get_standings_tool],
    prompt="You are a helpful assistant. The latest season is 2024 and we are in early 2025."
)

Let's test our agent with a simple query:

In [53]:
react_agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current standing of Sporting in the Primeira Liga?"}]}
)

INFO:__main__:Fetching standings for Primeira Liga (ID: 94) season 2024
INFO:__main__:Filtering for team: Sporting
INFO:__main__:Successfully retrieved standings data


{'messages': [HumanMessage(content='what is the current standing of Sporting in the Primeira Liga?', additional_kwargs={}, response_metadata={}, id='9314f67f-8bba-4d0a-9d41-b4e14dfb386b'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_standings', 'arguments': '{"team_name": "Sporting", "season": 2024.0, "league_name": "Primeira Liga"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--cbfd1e7c-3574-480d-856d-25d4d0f14f64-0', tool_calls=[{'name': 'get_standings', 'args': {'team_name': 'Sporting', 'season': 2024.0, 'league_name': 'Primeira Liga'}, 'id': '35135f48-64ed-4f83-bde8-7281c3491510', 'type': 'tool_call'}], usage_metadata={'input_tokens': 114, 'output_tokens': 16, 'total_tokens': 130, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="data={'get': 'standings', 'parameters': {'league': '94', 'season': '2024', 'team

And now a fake query to see how it handles errors:

In [54]:
react_agent.invoke(
    {"messages": [{"role": "user", "content": "what is the current standing of Sporting in the Eredivisie?"}]}
)

INFO:__main__:Fetching standings for Eredivisie (ID: 88) season 2024
INFO:__main__:Filtering for team: Sporting


{'messages': [HumanMessage(content='what is the current standing of Sporting in the Eredivisie?', additional_kwargs={}, response_metadata={}, id='91858a1b-9af9-45b7-b6d6-264cf199ae27'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_standings', 'arguments': '{"team_name": "Sporting", "season": 2024.0, "league_name": "Eredivisie"}'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run--edf74374-1600-42e2-843b-d7c3da8369d3-0', tool_calls=[{'name': 'get_standings', 'args': {'team_name': 'Sporting', 'season': 2024.0, 'league_name': 'Eredivisie'}, 'id': '441a19a4-5020-4713-95b2-af9c573887fa', 'type': 'tool_call'}], usage_metadata={'input_tokens': 114, 'output_tokens': 16, 'total_tokens': 130, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(content="error='No standings data found for Eredivisie season 2024' status_code=None", name='get_s

## 2. Upcoming fixtures

<https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `next` parameter

## 3. Last fixtures

<https://www.api-football.com/documentation-v3#tag/Fixtures/operation/get-fixtures> see `last` parameter

## 4. Specific fixtures

## 5. Match results

## 6. Match events

## 7. Player statistics

## Extra

### 8. Odds

### 9. Head-to-head (H2H)